In [10]:
import torch
print(torch.cuda.is_available())

True


In [11]:
import os
print(os.listdir("/kaggle/input"))

['datasets']


In [13]:
import os

base_path = "/kaggle/input"

for root, dirs, files in os.walk(base_path):
    level = root.replace(base_path, '').count(os.sep)
    if level <= 3:
        indent = ' ' * 2 * level
        print(f'{indent}{os.path.basename(root)}/')

input/
  datasets/
    mohammedabdeldayem/
      the-fake-or-real-dataset/


In [12]:
import os

base_path = "/kaggle/input/datasets/mohammedabdeldayem/the-fake-or-real-dataset"

for root, dirs, files in os.walk(base_path):
    level = root.replace(base_path, '').count(os.sep)
    if level <= 3:
        indent = ' ' * 2 * level
        print(f'{indent}{os.path.basename(root)}/ ({len(files)} files)')

the-fake-or-real-dataset/ (0 files)
  for-norm/ (0 files)
    for-norm/ (0 files)
      validation/ (0 files)
      training/ (0 files)
      testing/ (0 files)
  for-original/ (0 files)
    for-original/ (0 files)
      validation/ (0 files)
      training/ (0 files)
      testing/ (0 files)
  for-2sec/ (0 files)
    for-2seconds/ (0 files)
      validation/ (0 files)
      training/ (0 files)
      testing/ (0 files)
  for-rerec/ (0 files)
    for-rerecorded/ (0 files)
      validation/ (0 files)
      training/ (0 files)
      testing/ (0 files)


In [14]:
import os

base_path = "/kaggle/input/datasets/mohammedabdeldayem/the-fake-or-real-dataset/for-norm/for-norm/training"

real_files = os.listdir(base_path + "/real") if os.path.exists(base_path + "/real") else []
fake_files = os.listdir(base_path + "/fake") if os.path.exists(base_path + "/fake") else []

print(f"Real files: {len(real_files)}")
print(f"Fake files: {len(fake_files)}")
print("Sample real:", real_files[:2] if real_files else "EMPTY")
print("Sample fake:", fake_files[:2] if fake_files else "EMPTY")

Real files: 26941
Fake files: 26927
Sample real: ['file30736.wav_16k.wav_norm.wav_mono.wav_silence.wav', 'file25321.wav_16k.wav_norm.wav_mono.wav_silence.wav']
Sample fake: ['file14215.mp3.wav_16k.wav_norm.wav_mono.wav_silence.wav', 'file25706.mp3.wav_16k.wav_norm.wav_mono.wav_silence.wav']


In [15]:
!pip install librosa scikit-learn xgboost -q

In [16]:
import os
import numpy as np
import librosa
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from sklearn.model_selection import train_test_split
import pickle
import warnings

warnings.filterwarnings('ignore')

print("All libraries loaded!")

All libraries loaded!


In [17]:
def extract_features(file_path):
    try:
        y, sr = librosa.load(file_path, duration=3, sr=16000)
        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=40)
        features = np.mean(mfcc.T, axis=0)
        return features
    except:
        return None

print("Feature function ready!")

Feature function ready!


In [18]:
base_path = "/kaggle/input/datasets/mohammedabdeldayem/the-fake-or-real-dataset/for-norm/for-norm/training"

real_path = base_path + "/real"
fake_path = base_path + "/fake"

X, y = [], []

# Real files - 1000 only
real_files = os.listdir(real_path)[:1000]
for i, f in enumerate(real_files):
    feat = extract_features(os.path.join(real_path, f))
    if feat is not None:
        X.append(feat)
        y.append(0)  # 0 = real
    if i % 200 == 0:
        print(f"Real: {i}/1000 done...")

# Fake files - 1000 only
fake_files = os.listdir(fake_path)[:1000]
for i, f in enumerate(fake_files):
    feat = extract_features(os.path.join(fake_path, f))
    if feat is not None:
        X.append(feat)
        y.append(1)  # 1 = fake
    if i % 200 == 0:
        print(f"Fake: {i}/1000 done...")

X = np.array(X)
y = np.array(y)
print(f"\nDone! Total samples: {len(X)}")
print(f"Shape: {X.shape}")

Real: 0/1000 done...
Real: 200/1000 done...
Real: 400/1000 done...
Real: 600/1000 done...
Real: 800/1000 done...
Fake: 0/1000 done...
Fake: 200/1000 done...
Fake: 400/1000 done...
Fake: 600/1000 done...
Fake: 800/1000 done...

Done! Total samples: 2000
Shape: (2000, 40)


In [19]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

print("Model trained!")
print(f"Train samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")

Model trained!
Train samples: 1600
Test samples: 400


In [20]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

print(f"✅ Accuracy: {acc*100:.2f}%")
print(f"✅ F1 Score: {f1*100:.2f}%")
print(f"\nConfusion Matrix:")
print(cm)

✅ Accuracy: 96.00%
✅ F1 Score: 96.02%

Confusion Matrix:
[[191   8]
 [  8 193]]


In [21]:
from sklearn.metrics import roc_curve

fpr, tpr, thresholds = roc_curve(y_test, y_prob)
fnr = 1 - tpr

eer_idx = np.argmin(np.abs(fpr - fnr))
eer = (fpr[eer_idx] + fnr[eer_idx]) / 2

print(f"✅ EER: {eer*100:.2f}%")
print(f"✅ Required: ≤ 12%")
print(f"✅ Status: {'PASS ✅' if eer*100 <= 12 else 'FAIL ❌'}")

✅ EER: 4.25%
✅ Required: ≤ 12%
✅ Status: PASS ✅


In [23]:
import pickle

with open("deepfake_model.pkl", "wb") as f:
    pickle.dump(model, f)

print("✅ Model saved as deepfake_model.pkl")

✅ Model saved as deepfake_model.pkl


In [24]:
from IPython.display import FileLink
FileLink('deepfake_model.pkl')

/kaggle/working/deepfake_model.pkl

In [25]:
import shutil

# Ek real aur ek fake file copy karo
real_sample = os.path.join(real_path, os.listdir(real_path)[0])
fake_sample = os.path.join(fake_path, os.listdir(fake_path)[0])

shutil.copy(real_sample, "test_real.wav")
shutil.copy(fake_sample, "test_fake.wav")

from IPython.display import FileLink
display(FileLink('test_real.wav'))
display(FileLink('test_fake.wav'))

/kaggle/working/test_real.wav

/kaggle/working/test_fake.wav